# CISC7021 Group 38 — Member 3 (LLMLingua)
Before running: choose **Runtime > Change runtime type > T4 GPU**. This notebook reproduces the verified five-sample pilot at nominal 2x and 4x compression.

In [ ]:
!git clone https://github.com/jasonche515/CISC7021-Group38.git
%cd CISC7021-Group38
!python -m pip install -q -r part3_llmlingua/requirements.txt

## Stage 1: compress five fixed samples
Indices 17, 29, 126, 85 and 112 are the five shortest shared samples and fit the Phi-2 compressor context. Llama 2 is used only as a compatible tokenizer for measuring lengths; answer generation in Stage 2 still uses Llama 3.2.

In [ ]:
!python part3_llmlingua/run_llmlingua.py compress --sample-indices 17 29 126 85 112 --ratios 2 4 --tokenizer-model NousResearch/Llama-2-7b-hf --device cuda --jsonl part3_llmlingua/pilot5_compressed.jsonl --output part3_llmlingua/pilot5_compressed.json --no-resume

## Stage 2: load the target LLM and evaluate
The compression command has finished, so its process and GPU memory have already been released. Upgrade Transformers for Llama 3.2, then start evaluation.

In [ ]:
!python -m pip install -q -r part3_llmlingua/requirements_evaluate.txt

In [ ]:
!python part3_llmlingua/run_llmlingua.py evaluate --input part3_llmlingua/pilot5_compressed.jsonl --device cuda --dtype float16 --jsonl part3_llmlingua/pilot5_results.jsonl --output part3_llmlingua/pilot5_results.json --no-resume

## Show the five-sample pilot summary

In [ ]:
import json
with open('part3_llmlingua/pilot5_results.json', encoding='utf-8') as f:
    result = json.load(f)
result['summary']

## Download the result files
Keep both JSON and JSONL files so an interrupted run can be resumed later.

In [ ]:
from google.colab import files
for path in [
    'part3_llmlingua/pilot5_compressed.json',
    'part3_llmlingua/pilot5_compressed.jsonl',
    'part3_llmlingua/pilot5_results.json',
    'part3_llmlingua/pilot5_results.jsonl',
]:
    files.download(path)